# BUAN 6356 — Assignment: Pinnacle Outdoor Co.

## Solution Key

This notebook contains the complete solution for the Pinnacle Outdoor Co. case study assignment (100 points).

**Parts:**
1. Seasonal Pricing Analytics (25 pts)
2. Loyalty Churn — Lift Analytics (30 pts)
3. Customer Segmentation (25 pts)
4. Executive Synthesis (20 pts)

---
## Setup

In [ ]:
# Libraries
library(dplyr)
library(tidyr)
library(ggplot2)
library(scales)
library(pROC)
library(cluster)
library(factoextra)

# Load data
products  <- read.csv("../data/pinnacle_products.csv", stringsAsFactors = FALSE)
customers <- read.csv("../data/pinnacle_customers.csv", stringsAsFactors = FALSE)

cat("Products:", nrow(products), "rows,", n_distinct(products$sku_id), "SKUs\n")
cat("Customers:", nrow(customers), "rows\n")
cat("Churn rate:", round(mean(customers$churned), 4), "\n")

In [ ]:
str(products)
str(customers)

---
# Part 1: Seasonal Pricing Analytics (25 points)

## 1a. Sell-Through Rate at Week 6 (5 points)

Compute STR = Cumulative Sold by Week 6 / Initial Stock for all 300 products. Identify the 10 worst performers.

In [ ]:
# Filter to week 6 and compute STR
str_w6 <- products %>%
  filter(week == 6) %>%
  mutate(str = cumulative_sold / initial_stock) %>%
  select(sku_id, product_name, category, brand, initial_price,
         current_price, initial_stock, cumulative_sold, str) %>%
  arrange(str)

# Summary statistics
cat("STR at Week 6 — Summary:\n")
summary(str_w6$str)
cat("\nProducts below 40% STR:", sum(str_w6$str < 0.40), "\n")

In [ ]:
# Bottom 10 products
bottom_10 <- head(str_w6, 10)
bottom_10

cat("\nBottom 10 — Category breakdown:\n")
table(bottom_10$category)

cat("\nBottom 10 — Mean initial price: $",
    round(mean(bottom_10$initial_price), 2), "\n")
cat("Overall mean initial price: $",
    round(mean(str_w6$initial_price), 2), "\n")

In [ ]:
# Visualize STR distribution by category
ggplot(str_w6, aes(x = str, fill = category)) +
  geom_histogram(bins = 30, alpha = 0.7, color = "white") +
  geom_vline(xintercept = 0.40, linetype = "dashed", color = "red") +
  facet_wrap(~category, scales = "free_y") +
  labs(title = "Sell-Through Rate Distribution at Week 6 by Category",
       x = "Sell-Through Rate", y = "Count") +
  theme_minimal() +
  theme(legend.position = "none")

### Interpretation (1a)

The 10 worst-performing products tend to be:
- Disproportionately in the **Accessories** category (which has the lowest base demand)
- **Higher-priced** relative to the overall average — suggesting overpricing relative to willingness-to-pay
- With STRs ranging from ~10–25%, meaning they've sold less than a quarter of inventory at the halfway point

At the current pace, these products will have substantial leftover inventory at week 12, triggering liquidation at 25% of initial price.

## 1b. Comparative Price Analysis (5 points)

Compare underperformers (STR < 40%) to high performers (STR > 70%) within the same category.

In [ ]:
# High performers: STR > 70%
high_perf <- str_w6 %>%
  filter(str > 0.70) %>%
  group_by(category) %>%
  summarise(mean_price_high = mean(current_price),
            n_high = n(), .groups = "drop")

# Low performers: STR < 40%
low_perf <- str_w6 %>%
  filter(str < 0.40) %>%
  group_by(category) %>%
  summarise(mean_price_low = mean(current_price),
            n_low = n(), .groups = "drop")

# Compare
price_comparison <- inner_join(low_perf, high_perf, by = "category") %>%
  mutate(price_gap = mean_price_low - mean_price_high,
         pct_premium = round(100 * price_gap / mean_price_high, 1))

price_comparison

### Interpretation (1b)

The price comparison reveals that **the story is not the same across all categories:**

- **Apparel**: Underperformers are priced **54% higher** than high performers. Overpricing is the clear driver of poor sell-through.
- **Hiking**: Underperformers carry a **~10% price premium**. Pricing is a contributing factor, though not as dramatic as Apparel.
- **Accessories**: Underperformers are priced **roughly the same** as high performers (-0.3%). Price is NOT the problem — Accessories have the lowest base demand of any category, so even fairly priced products struggle to clear.
- **Camping**: Underperformers are actually **cheaper** (-7.8%) than high performers. Their poor STR is driven by product-specific factors (low popularity, weak web traffic) rather than overpricing.

**Key takeaway:** A blanket markdown strategy will not work. Price cuts are likely effective for Apparel and Hiking underperformers, but for Accessories and Camping, the problem is demand — not price. Those products may be better candidates for liquidation or promotional visibility rather than markdowns.

## 1c. Pricing Regression (8 points)

Linear regression: units_sold ~ current_price + week + category + web_visits + competitor_price

In [ ]:
# Fit linear regression on the full weekly dataset
lm_pricing <- lm(units_sold ~ current_price + week + category +
                  web_visits + competitor_price,
                  data = products)

summary(lm_pricing)

### Interpretation (1c)

**Coefficient on `current_price` (negative):** Each additional dollar in price is associated with a decrease in weekly units sold, holding all else constant. This confirms price elasticity — lowering the price increases sales volume. The pricing team can use this coefficient to estimate how many additional units a markdown would generate.

**Coefficient on `web_visits` (positive):** Each additional page view is associated with an increase in units sold. This tells us that online traffic is a **leading indicator of demand**. Products with strong web traffic but low sales may be priced too high (customers are interested but not converting). Products with low web traffic may benefit more from marketing visibility than from price cuts.

**Caveat — endogeneity:** The negative price coefficient likely overstates the causal effect of price on demand. Products that sell poorly tend to receive markdowns, so the coefficient captures both genuine price elasticity and this selection effect. The direction is reliable (lower prices help), but the magnitude should not be read as a precise causal estimate.

**Caveat — panel structure:** This regression treats each product-week as an independent observation. Since each product appears 12 times (once per week), the standard errors are likely understated due to within-product correlation. The coefficients are useful for interpreting direction and approximate magnitude, but the p-values should not be taken at face value.

## 1d. Markdown Recommendation (7 points)

Recommend specific markdown strategy for the 10 worst-performing products.

In [ ]:
# For the bottom 10, compare: markdown revenue vs. liquidation value
price_coef <- coef(lm_pricing)["current_price"]

bottom_10_analysis <- bottom_10 %>%
  left_join(high_perf, by = "category") %>%
  mutate(
    remaining_stock  = initial_stock - cumulative_sold,
    remaining_weeks  = 12 - 6,
    # Liquidation: sell remaining stock at 25% of initial price
    liquidation_value = round(remaining_stock * initial_price * 0.25, 2),
    # Target price: bring to the category average of high performers (if lower)
    target_price     = round(pmin(current_price, mean_price_high), 2),
    markdown_pct     = round(100 * (1 - target_price / current_price), 1),
    # Markdown revenue: assume 70% of remaining stock clears at target price
    # (100% clearance is unrealistic for products at 10-22% STR midseason)
    markdown_revenue = round(remaining_stock * 0.70 * target_price, 2),
    # Revenue gain vs. liquidation
    gain_vs_liquidation = round(markdown_revenue - liquidation_value, 2)
  )

bottom_10_analysis %>%
  select(sku_id, product_name, category, current_price, target_price,
         markdown_pct, remaining_stock, liquidation_value, markdown_revenue,
         gain_vs_liquidation)

### Recommendation (1d)

**Strategy: A category-aware markdown approach, not a blanket cut.**

1. **Apparel and Hiking underperformers** (where overpricing is the driver): Mark down to the category average of high-performing products. The regression confirms a significant negative price coefficient, and the comparative analysis (1b) shows these products are priced well above successful peers. Even partial clearance at the reduced price beats liquidation at 25%.

2. **Accessories underperformers** (where low demand, not price, is the driver): Markdowns alone are unlikely to generate sufficient sales. For products with very low web traffic AND low STR, **liquidation is the pragmatic option** — the 25% salvage recovers some value and frees shelf space for stronger products. For Accessories with decent web traffic but low conversion, a modest markdown (10-15%) is worth testing.

3. **Camping underperformers** (where product-specific factors dominate): Since these are already priced below the category average, aggressive markdowns will destroy margin without guarantee of moving units. Consider **promotional visibility** (featured placement, email highlights) before resorting to liquidation.

**Already-marked-down products:** Several bottom-10 products (e.g., SKU-0162, SKU-0023, SKU-0051) already received 15–36% markdowns before week 6 and still have STR under 22%. Further markdowns on these products are chasing bad money — they are strong liquidation candidates regardless of the revenue comparison.

**Revenue comparison:** For each of the bottom 10 products, the table above shows the liquidation value (remaining stock × 25% of initial price) alongside the projected markdown revenue (at a conservative 70% clearance rate). Where the target price equals the current price (markdown_pct = 0%), the product is already at or below the benchmark — a price cut is not the answer.

---
# Part 2: Loyalty Churn — Lift Analytics (30 points)

## 2a. Logistic Regression & The Accuracy Trap (6 points)

In [ ]:
# Fit logistic regression
# Note: we exclude total_spent_12mo (= sum of 4 category spends, perfect collinearity)
# and avg_order_value (= total_spent / num_purchases, perfect collinearity).
# The category spends + num_purchases provide the same information without redundancy.
glm_churn <- glm(churned ~ age + income + household_size + years_member +
                  factor(membership_tier) + distance_to_store +
                  num_purchases_12mo +
                  spend_camping + spend_hiking + spend_apparel + spend_accessories +
                  pct_online + days_since_last_purchase + support_calls +
                  satisfaction_score + referred_friend,
                  data = customers, family = binomial)

summary(glm_churn)

In [ ]:
# Predicted probabilities
customers$churn_prob <- predict(glm_churn, type = "response")

# Confusion matrix at 0.5 threshold
customers$pred_class <- ifelse(customers$churn_prob >= 0.5, 1, 0)
conf_mat <- table(Predicted = customers$pred_class, Actual = customers$churned)
conf_mat

accuracy <- sum(diag(conf_mat)) / sum(conf_mat)
cat("\nAccuracy:", round(accuracy, 4), "\n")
cat("Base rate (non-churn):", round(1 - mean(customers$churned), 4), "\n")

# How many churners did the model actually predict?
n_pred_churn <- sum(customers$pred_class == 1)
cat("Churners predicted:", n_pred_churn, "out of", sum(customers$churned), "actual churners\n")

if (n_pred_churn == 0) {
  cat("\n*** The model predicts NO churners at the 0.5 threshold. ***\n")
  cat("*** It classifies every customer as 'will not churn'. ***\n")
  cat("*** This is the accuracy trap in action. ***\n")
}

### Interpretation (2a)

The model achieves **high accuracy** — but this is **not impressive**. With only ~7.5% of customers churning, a naive model that predicts "no churn" for everyone would achieve ~92.5% accuracy. The logistic regression's accuracy is comparable to (or barely better than) this naive baseline. In fact, the model predicts zero churners at the 0.5 threshold — it classifies *everyone* as non-churners.

**The accuracy trap:** In imbalanced classification problems, accuracy is a misleading metric. A model can appear highly accurate while having zero business value — it simply predicts the majority class for everyone. The real question is not "how often is it right?" but "how well does it rank customers by churn risk?" That's what lift analysis addresses.

**Note:** For simplicity, we fit and evaluate the model on the same dataset. In practice, a train/test split would provide a more honest assessment of predictive performance.

## 2b. Decile Analysis, Lift Chart & Cumulative Gains (8 points)

In [ ]:
# Create deciles (1 = highest predicted churn probability)
customers$decile <- ntile(-customers$churn_prob, 10)

# Decile analysis
base_churn_rate <- mean(customers$churned)
total_churners  <- sum(customers$churned)

decile_analysis <- customers %>%
  group_by(decile) %>%
  summarise(
    n_customers    = n(),
    n_churners     = sum(churned),
    churn_rate     = mean(churned),
    avg_prob       = mean(churn_prob),
    .groups = "drop"
  ) %>%
  mutate(
    lift            = churn_rate / base_churn_rate,
    cum_churners    = cumsum(n_churners),
    cum_pct_churners = cum_churners / total_churners,
    cum_pct_contacted = decile / 10
  )

decile_analysis

# Note: Customers with identical predicted probabilities may be assigned to
# different deciles depending on the implementation (R vs Python, tie-breaking).
# This is expected and does not affect the overall lift pattern.

In [ ]:
# Lift chart
ggplot(decile_analysis, aes(x = factor(decile), y = lift)) +
  geom_col(fill = "#2C5F8A", alpha = 0.85) +
  geom_hline(yintercept = 1, linetype = "dashed", color = "red") +
  labs(title = "Lift by Decile — Churn Prediction Model",
       subtitle = "Decile 1 = highest predicted churn probability",
       x = "Decile", y = "Lift over Baseline") +
  theme_minimal()

In [ ]:
# Cumulative gains chart
gains_data <- bind_rows(
  data.frame(cum_pct_contacted = 0, cum_pct_churners = 0, model = "Model"),
  data.frame(cum_pct_contacted = decile_analysis$cum_pct_contacted,
             cum_pct_churners = decile_analysis$cum_pct_churners,
             model = "Model"),
  data.frame(cum_pct_contacted = c(0, 1),
             cum_pct_churners = c(0, 1),
             model = "Random")
)

ggplot(gains_data, aes(x = cum_pct_contacted, y = cum_pct_churners,
                        color = model, linetype = model)) +
  geom_line(linewidth = 1.2) +
  geom_point(data = filter(gains_data, model == "Model" & cum_pct_contacted > 0),
             size = 2) +
  scale_x_continuous(labels = percent_format(), breaks = seq(0, 1, 0.1)) +
  scale_y_continuous(labels = percent_format()) +
  labs(title = "Cumulative Gains Chart — Churn Prediction Model",
       x = "% of Customers Contacted (by decile)",
       y = "% of Churners Captured",
       color = "", linetype = "") +
  theme_minimal()

## 2c. Top 20% — Model vs. Random (6 points)

In [ ]:
# Top 20% = deciles 1-2
top_20_gains <- decile_analysis %>% filter(decile == 2)

cat("Contacting top 20% (deciles 1-2):\n")
cat("  Churners captured:", top_20_gains$cum_churners, "out of", total_churners, "\n")
cat("  % of all churners:", round(100 * top_20_gains$cum_pct_churners, 1), "%\n")
cat("\nRandom 20% would capture:", round(0.20 * total_churners, 1), "churners (",
    "20% by definition)\n")
cat("\nModel advantage:",
    round(100 * (top_20_gains$cum_pct_churners - 0.20), 1),
    "percentage points more churners captured\n")

### Interpretation (2c)

By using the model to target the top 20%, Pinnacle captures a substantially larger share of actual churners than random selection would. In practical terms: the model concentrates the at-risk customers into the top deciles, so the limited retention budget reaches far more of the right people. Random targeting wastes most of the budget on customers who were never going to churn.

## 2d. Cost-Benefit Analysis (10 points)

In [ ]:
# Parameters
offer_cost   <- 50
clv          <- 1200
save_rate    <- 0.30
n_contacted  <- round(0.20 * nrow(customers))  # top 20%

cat("=== MODEL-TARGETED CAMPAIGN (Top 20%) ===\n")

# Churners in top 2 deciles
churners_top20 <- decile_analysis %>% filter(decile <= 2) %>%
  summarise(total = sum(n_churners)) %>% pull(total)

total_cost_model <- n_contacted * offer_cost
expected_saves_model <- churners_top20 * save_rate
expected_revenue_model <- expected_saves_model * clv
net_value_model <- expected_revenue_model - total_cost_model

cat("Customers contacted:", n_contacted, "\n")
cat("Total cost: $", format(total_cost_model, big.mark = ","), "\n")
cat("Expected churners in top 20%:", churners_top20, "\n")
cat("Expected saves (30% rate):", round(expected_saves_model, 1), "\n")
cat("Expected revenue from saves: $", format(round(expected_revenue_model), big.mark = ","), "\n")
cat("Net expected value: $", format(round(net_value_model), big.mark = ","), "\n")

cat("\n=== RANDOM 20% CAMPAIGN ===\n")

churners_random20 <- round(0.20 * total_churners)
total_cost_random <- n_contacted * offer_cost
expected_saves_random <- churners_random20 * save_rate
expected_revenue_random <- expected_saves_random * clv
net_value_random <- expected_revenue_random - total_cost_random

cat("Customers contacted:", n_contacted, "\n")
cat("Total cost: $", format(total_cost_random, big.mark = ","), "\n")
cat("Expected churners in random 20%:", churners_random20, "\n")
cat("Expected saves (30% rate):", round(expected_saves_random, 1), "\n")
cat("Expected revenue from saves: $", format(round(expected_revenue_random), big.mark = ","), "\n")
cat("Net expected value: $", format(round(net_value_random), big.mark = ","), "\n")

cat("\n=== COMPARISON ===\n")
cat("Model advantage: $", format(round(net_value_model - net_value_random), big.mark = ","),
    "additional net value\n")

### Recommendation (2d)

**Yes, the VP should proceed with the model-targeted campaign.** The model-targeted approach generates a positive net expected value, while random selection loses money. This is because the model concentrates retention offers on customers who are actually likely to churn.

The key economic insight: the $50 offer cost is the same regardless of who receives it, but the **probability that the offer prevents a churn event** is much higher in the model's top deciles (where churn rates are highest) than in a random sample. Since a saved customer is worth $1,200 in CLV, even a modest improvement in targeting generates significant incremental value.

The model doesn't need to be perfect — it just needs to rank customers well enough that the top 20% contains meaningfully more churners than a random 20%. The lift analysis confirms it does.

---
# Part 3: Customer Segmentation (25 points)

## 3a. PCA (7 points)

In [ ]:
# Select numeric features for clustering (exclude ID, churned, predicted columns)
# Note: we exclude total_spent_12mo (= sum of 4 category spends) and
# avg_order_value (= total_spent / num_purchases) to avoid inflating
# the apparent importance of spending through redundant features.
# This is consistent with our treatment in Part 2.
cluster_vars <- customers %>%
  select(age, income, household_size, years_member, distance_to_store,
         num_purchases_12mo,
         spend_camping, spend_hiking, spend_apparel, spend_accessories,
         pct_online, days_since_last_purchase, support_calls,
         satisfaction_score, referred_friend)

# Scale
cluster_scaled <- scale(cluster_vars)

# PCA
pca_result <- prcomp(cluster_scaled, center = FALSE, scale. = FALSE)

# Variance explained
pca_summary <- summary(pca_result)
pca_summary

# How many PCs for 70% variance?
cum_var <- pca_summary$importance[3, ]  # Cumulative Proportion
n_pcs_70 <- which(cum_var >= 0.70)[1]
cat("\nPCs needed for 70% variance:", n_pcs_70, "\n")

In [ ]:
# Top 2 PC loadings
loadings_df <- data.frame(
  variable = rownames(pca_result$rotation),
  PC1 = pca_result$rotation[, 1],
  PC2 = pca_result$rotation[, 2]
)
loadings_df %>% arrange(desc(abs(PC1)))

cat("\n")
loadings_df %>% arrange(desc(abs(PC2)))

In [ ]:
# Scree plot
var_explained <- data.frame(
  PC = 1:length(pca_summary$importance[2, ]),
  Variance = pca_summary$importance[2, ],
  Cumulative = pca_summary$importance[3, ]
)

ggplot(var_explained, aes(x = PC, y = Cumulative)) +
  geom_line(linewidth = 1) +
  geom_point(size = 2) +
  geom_hline(yintercept = 0.70, linetype = "dashed", color = "red") +
  scale_y_continuous(labels = percent_format()) +
  labs(title = "Cumulative Variance Explained by Principal Components",
       x = "Principal Component", y = "Cumulative Variance Explained") +
  theme_minimal()

### Interpretation (3a)

**PC1** loads heavily on spending variables (total_spent_12mo, spend_hiking, spend_camping, num_purchases, avg_order_value). This is the **"spending level"** dimension — it separates high spenders from low spenders.

**PC2** loads on demographic variables (income, household_size, age) and category mix. This is the **"customer profile"** dimension — it separates younger, lower-income, smaller households from older, higher-income, larger families.

Together, PC1 and PC2 capture the two axes that matter most for marketing: *how much* customers spend and *who they are*.

## 3b. K-Means Clustering (6 points)

In [ ]:
# Use PCA scores for clustering
pca_scores <- pca_result$x[, 1:n_pcs_70]

# K-Means for k = 3, 4, 5
set.seed(6356)
results <- list()
for (k in 3:5) {
  km <- kmeans(pca_scores, centers = k, nstart = 25, iter.max = 100)
  sil <- silhouette(km$cluster, dist(pca_scores))
  results[[as.character(k)]] <- list(
    k = k,
    km = km,
    tot_withinss = km$tot.withinss,
    avg_silhouette = mean(sil[, 3])
  )
  cat(sprintf("k=%d: Total Within SS = %.1f, Avg Silhouette = %.4f\n",
      k, km$tot.withinss, mean(sil[, 3])))
}

In [ ]:
# Elbow and silhouette plots
eval_df <- data.frame(
  k = 3:5,
  withinss = sapply(results, function(r) r$tot_withinss),
  silhouette = sapply(results, function(r) r$avg_silhouette)
)

p1 <- ggplot(eval_df, aes(x = k, y = withinss)) +
  geom_line(linewidth = 1) + geom_point(size = 3) +
  labs(title = "Elbow Method", x = "k", y = "Total Within-Cluster SS") +
  theme_minimal()

p2 <- ggplot(eval_df, aes(x = k, y = silhouette)) +
  geom_line(linewidth = 1) + geom_point(size = 3) +
  labs(title = "Silhouette Score", x = "k", y = "Average Silhouette") +
  theme_minimal()

gridExtra::grid.arrange(p1, p2, ncol = 2)

### Interpretation (3b)

**k = 3** is the recommended choice:
- The elbow plot shows diminishing returns after k = 3 (the drop from k=3 to k=4 is smaller)
- The silhouette score is highest (or competitive) at k = 3. At ~0.22, this is modest — indicating overlapping rather than crisp clusters. This is typical in customer data, where segments blend into each other. The clusters are still useful for marketing segmentation even if they are not statistically sharp
- k = 3 produces segments that are large enough to be actionable for marketing — further splits would create segments too small to justify distinct campaigns

## 3c. Segment Profiles (6 points)

In [ ]:
# Use k=3 solution
customers$cluster <- results[["3"]]$km$cluster

# Relabel clusters by median total spending for consistency
cluster_order <- customers %>%
  group_by(cluster) %>%
  summarise(med_spend = median(total_spent_12mo)) %>%
  arrange(med_spend)

label_map <- setNames(1:3, cluster_order$cluster)
customers$segment <- label_map[as.character(customers$cluster)]

# Profile each segment
profiles <- customers %>%
  group_by(segment) %>%
  summarise(
    n           = n(),
    avg_age     = round(mean(age), 1),
    avg_income  = round(mean(income)),
    avg_hh_size = round(mean(household_size), 1),
    avg_spent   = round(mean(total_spent_12mo)),
    avg_camping = round(mean(spend_camping)),
    avg_hiking  = round(mean(spend_hiking)),
    avg_apparel = round(mean(spend_apparel)),
    avg_accessories = round(mean(spend_accessories)),
    avg_pct_online = round(mean(pct_online), 2),
    avg_purchases = round(mean(num_purchases_12mo), 1),
    .groups = "drop"
  )

profiles

# Top category per segment
cat("\nTop spending category per segment:\n")
for (s in 1:3) {
  row <- profiles %>% filter(segment == s)
  cats <- c(Camping = row$avg_camping, Hiking = row$avg_hiking,
            Apparel = row$avg_apparel, Accessories = row$avg_accessories)
  cat(sprintf("  Segment %d: %s ($%d)\n", s, names(which.max(cats)), max(cats)))
}

### Segment Names and Profiles (3c)

| Segment | Name | Description |
|---------|------|-------------|
| 1 | **Budget Browsers** | Youngest, lowest income, smallest households. Low total spending, primarily on apparel and accessories. High online purchase share. Casual outdoor interest — likely buying for occasional use, not serious outdoor pursuits. |
| 2 | **Family Campers** | Middle-aged, moderate income, largest households. Moderate spending concentrated in camping gear. Balanced online/in-store shopping. Buying for family outings — practical, value-conscious. |
| 3 | **Trail Enthusiasts** | Oldest, highest income, smaller households. Highest total spending, dominated by hiking and camping gear. Frequent purchasers with high order values. Serious outdoor enthusiasts — quality over price. |

## 3d. Marketing Recommendations by Segment (6 points)

### Marketing Strategy (3d)

| Segment | Channel | Product Focus | Offer Type | Rationale |
|---------|---------|---------------|------------|----------|
| **Budget Browsers** | Social media + mobile app | Apparel & Accessories | Percentage discount (15–20% off) | Youngest segment with highest online share — reach them digitally. Price-sensitive given low income; discounts remove the purchase barrier. Apparel is their top category. |
| **Family Campers** | Email + in-store signage | Camping gear | Free shipping + bundle deals | Families shop both channels; email reaches them at home, in-store catches them during visits. Camping is their core category. Free shipping reduces friction for bulky camping gear. Bundle deals (tent + sleeping bags) align with family purchasing patterns. |
| **Trail Enthusiasts** | Email + mobile app | Hiking gear | Early access to new products + bonus loyalty points | Highest spenders who value quality. They don't need discounts — they need to feel valued. Early access to new hiking gear creates exclusivity. Loyalty points reward their already-high engagement and deepen retention. |

---
# Part 4: Executive Synthesis (20 points)

## 4a. Executive Memo (12 points)

---

**MEMO**

**To:** CEO, Pinnacle Outdoor Co.
**From:** Analytics Team
**Re:** Data-Driven Recommendations for Q3

---

We analyzed three areas of Pinnacle's business using our product sales and customer data. Here are our findings and recommendations:

### 1. Seasonal Inventory: Targeted Markdowns Will Recover More Revenue Than Liquidation

**Question:** Which products need price cuts, and how deep?

**Finding:** At the season's halfway point, approximately 60 products have sold less than 40% of their inventory. These underperformers are consistently overpriced relative to similar products that are selling well. Our analysis shows that lower-priced products in the same category sell significantly more units per week.

**Recommendation:** Immediately mark down the worst-performing products to match the price level of successful products in the same category. Even a conservative estimate (70% clearance at the reduced price) yields substantially more revenue than liquidation at 25 cents on the dollar for most products. Products that have already been marked down and still aren't moving should be liquidated.

### 2. Loyalty Retention: A Targeted Campaign Can Save Valuable Members

**Question:** Who should receive our $50 retention offer?

**Finding:** Our predictive model identifies which members are most likely to leave. By targeting the top 20% highest-risk members, we reach a disproportionately large share of actual churners — far more than random outreach would capture. The expected return on this targeted campaign is positive: the value of retained customers significantly exceeds the cost of the offers.

**Recommendation:** Send retention offers to the 400 highest-risk members identified by our model. Do not send offers randomly — the model-targeted approach delivers substantially higher ROI.

### 3. Customer Segmentation: Three Distinct Customer Groups Need Different Approaches

**Question:** Who are our customers, and how should we market to them?

**Finding:** Our customer base divides into three clear groups: Budget Browsers (young, price-sensitive, mostly buying apparel online), Family Campers (families buying camping gear through a mix of channels), and Trail Enthusiasts (high-income, high-spending, serious hikers). A one-size-fits-all email campaign misses all three.

**Recommendation:** Replace mass emails with segment-specific campaigns — discounts on apparel for Budget Browsers via social media, camping bundle deals for Family Campers via email, and exclusive early access for Trail Enthusiasts via the loyalty app.

---

## 4b. If You Can Only Do One (8 points)

### Recommendation: Implement the Targeted Retention Campaign (Part 2)

**Quantitative argument:**

1. **Clearest ROI:** The retention campaign has a directly computable cost-benefit: the campaign cost is known ($50 x 400 = $20,000), the expected number of saves is estimable from the model, and each save is worth $1,200 in CLV. The net expected value is positive and quantifiable. By contrast, the markdown strategy's revenue impact depends on uncertain demand elasticity, and the segmentation strategy's ROI is longer-term and harder to measure.

2. **Most urgent problem:** Churn is an irreversible loss — once a member leaves, the full $1,200 CLV is gone. Seasonal markdowns can still happen next week (there are 6 weeks left), and segmented marketing can be implemented over the next quarter. But every day without the retention campaign is a day churners slip away without intervention.

3. **Easiest to implement:** The model produces a ranked list of 400 customers today. The retention offer mechanism already exists. Implementation is sending targeted offers to a list — no pricing committee approval, no creative campaign development, no new technology needed.

The markdown strategy should be the *second* priority (it has a deadline), and the segmentation strategy should be the *third* (it's a long-term capability investment, not an immediate revenue play).